### A/B Test Design: Contextual AI Assistant Prompt to Improve Browser Retention

### **Business Context**


Edge ships pre-installed on new Windows devices, but installation doesn't guarantee that users stick with it. The core question is simple: if a new user sees something genuinely useful early — like a prompt that summarizes the page they're reading — does that give them a reason to make Edge their default browser?

This experiment tests that mechanism. The treatment is a single contextual Copilot prompt ("Summarize this page") shown on the first high-content page a user visits, once per session, for up to 3 sessions.

**Context filter:** prompt only fires on high-content pages — content length above threshold, non-search domains, meaningful scroll depth. Firing on login pages or search results inflates dismissals and underestimates Copilot's real value.

We check two things:
- Within the first week: did users who saw the prompt set Edge as their default browser?
- At day 30: what share of their total browsing time is spent in Edge vs other browsers?

The first question tells us whether the prompt drives immediate action. The second tells us whether that action translated into genuine habit.


### **Hypothesis**

If we surface a contextual Copilot prompt ("Summarize this page") once per session — on the first high-content page visit — for up to 3 sessions on a new device, then 7-day default browser set rate will increase by ~1.5–2pp, because users discover differentiated value early — before browser habits have formed — and are more likely to set Edge as their default.

- **H0:** 7-day default browser set rate is equal across treatment and control
- **H1:** 7-day default browser set rate differs between groups (two-tailed — prompt could backfire if it feels intrusive)
- **MDE:** powered for 1.5pp; 2pp is the optimistic hypothesis

Design note: With up to 3 exposures per device, session timing becomes a confound — a device that sees the prompt in session 1 is in a different habit-formation state than one that sees it first in session 3. In a follow-up experiment, session of first exposure should be treated as a covariate or used to define subgroups. For this experiment, we acknowledge the confound and interpret results as the average effect across all exposure timings.


### **Metric Structure**

**Diagnostic**
- Copilot activation rate — used Copilot ≥1 time in first 7 days. Not a success metric. If primary doesn't move: activation low → prompt didn't reach users, fix the surface. Activation high → Copilot usage isn't driving default-set behavior, reconsider hypothesis.
- Dismissal rate — proportion closing prompt within 5s with no interaction. High dismissals = UX surface is wrong, not necessarily a ship-stopper on its own.

**Primary**
- 7-day default browser set rate — proportion of new devices where Edge is set as the default browser within 7 days of initial setup. Ship/no-ship decision is made on this metric.

**Secondary (exploratory — pre-specified, not decision-making)**
- Default retention at day 30 — among devices that set Edge as default by day 7, what % still have it set at day 30? Separates genuine preference from initial inertia.
- Share of browsing time at day 30 — Edge browsing minutes ÷ total browser minutes on device. Checks whether default-set translated into actual usage habit. Measurable on Windows where OS-level telemetry can observe all browser activity.
- Copilot depth — # sessions, tasks per active device

**Guardrails**
- Search queries per device — hard guardrail. If Copilot substitutes for search, query volume drops. Direct revenue risk. Require non-negative impact on search queries (or pre-agreed acceptable loss) for rollout, even if default-set rate improves.
- Crash rate, page load latency
- Uninstall rate

**Note on primary metric:**
7-day default browser set rate is a binary device-level outcome — Edge is set as the default browser within 7 days of setup (1) or not (0). Share of browsing time captures the behavioral dimension separately — whether the user is actually spending time in Edge vs other browsers. Both are tracked: default-set rate as the primary decision metric, share of browsing time as the post-ship durability check.


### **Target Population**



New Windows devices, up to first 3 Edge sessions post-setup. Prompt fires at most once per session on the first high-content page visit. Excluding devices where a non-Edge default was set before initial setup completed.

#### **Group Definitions**

Control: new devices, initial setup completed, no Copilot prompt shown
Treatment: new devices, initial setup completed, Copilot "Summarize this page" prompt shown once on first high-content page visit, for up to 3 sessions

Outcome measured per device at day 7:
1 = Edge is set as the default browser
0 = Edge is not set as the default browser

Both groups exclude:
- devices where a non-Edge default was set before setup completed
- devices that never reached a high-content page (ineligible for exposure)


### **Randomization Unit**

**Device-level:** browser retention is tracked per device.


### **Evaluation Windows**

- Days 1–3: stability check — assignment balance, crash rate. Start at 5% ramp until clean.
- Day 7: primary decision window — default browser set rate. Z-test run here. Ship/no-ship decision made on this read.
- Day 30: post-ship durability — share of browsing time (Edge minutes ÷ total browser minutes), default retention rate, search queries per device. Not a ship gate — confirms whether the D7 effect translated into genuine browsing habit.

#### **Causal Chain**

assignment → Copilot prompt exposure → Copilot use → default browser set → share of browsing time

This chain is acknowledged but not fully modeled in this notebook — see Next Steps.


### **Statistical Test**

**Test:** Two-proportion z-test
- Metric is binary per device — set Edge as default browser by day 7 (1) or not (0)
- Two independent groups — treatment and control devices never overlap (device-level randomization)
- Large sample expected — z-test is appropriate


### **Power Analysis**

In [ ]:
import math
from statsmodels.stats.proportion import proportion_effectsize
from statsmodels.stats.power import NormalIndPower

# Parameters
p_baseline  = 0.55
mde_pp      = 0.015
p_treatment = p_baseline + mde_pp
alpha       = 0.05
power       = 0.80
daily_pop   = 50_000
ramp_pct    = 0.10

# Cohen's h — abs() so sign never leaks into downstream calculations or prints
effect_size = abs(proportion_effectsize(p_baseline, p_treatment))

analysis = NormalIndPower()
n_per_variant = math.ceil(
    analysis.solve_power(effect_size=effect_size, alpha=alpha, power=power, alternative='two-sided')
)

n_total          = n_per_variant * 2
daily_eligible   = daily_pop * ramp_pct
days_to_enroll   = math.ceil(n_total / daily_eligible)
decision_window  = 7   # primary metric read at day 7
total_duration   = days_to_enroll + decision_window

print("=" * 55)
print("  A/B TEST POWER ANALYSIS — 7-DAY DEFAULT BROWSER SET")
print("=" * 55)
print(f"\nINPUTS")
print(f"  Baseline default-set rate  : {p_baseline:.1%}")
print(f"  Treatment rate (MDE)       : {p_treatment:.1%}")
print(f"  MDE                        : +{mde_pp:.1%} (absolute)")
print(f"  Alpha (two-tailed)         : {alpha}")
print(f"  Power                      : {power:.0%}")
print(f"  Daily new devices          : {daily_pop:,}")
print(f"  Experiment ramp            : {ramp_pct:.0%}")
print(f"  Daily eligible devices     : {daily_eligible:,.0f}")
print(f"\nEFFECT SIZE")
print(f"  Cohen's h                  : {effect_size:.4f}")
print(f"\nRESULTS")
print(f"  Sample size per variant    : {n_per_variant:,}")
print(f"  Total sample size          : {n_total:,}")
print(f"  Days to enroll             : {days_to_enroll}")
print(f"  Decision window            : {decision_window} days")
print(f"  Total experiment clock     : {total_duration} days")
print(f"  Post-ship durability read  : day 30 (share of browsing time)")
print("=" * 55)

print()
print(f"{'MDE':>6}  {'Treatment p':>12}  {'Cohen h':>9}  {'n/variant':>10}  {'Days':>6}")
print("-" * 50)
for d in [0.010, 0.015, 0.020, 0.025]:
    p2   = p_baseline + d
    h    = abs(proportion_effectsize(p_baseline, p2))
    n    = math.ceil(analysis.solve_power(h, alpha=alpha, power=power, alternative='two-sided'))
    days = math.ceil(n * 2 / daily_eligible)
    print(f"{d*100:>5.1f}pp  {p2:>12.3f}  {h:>9.4f}  {n:>10,}  {days:>6}")


### **Data Simulation**

In [23]:
import numpy as np

rng = np.random.default_rng(seed=42)

# Ground truth
n_per_variant   = 17_210
p_control       = 0.55    # baseline 7-day default browser set rate
p_treatment     = 0.565   # baseline + 1.5pp true effect
true_lift       = p_treatment - p_control

# Binary outcome per device: 1 = Edge set as default browser by day 7, 0 = not
control_outcomes   = rng.binomial(n=1, p=p_control,   size=n_per_variant)
treatment_outcomes = rng.binomial(n=1, p=p_treatment, size=n_per_variant)

# Observed rates
obs_control   = control_outcomes.mean()
obs_treatment = treatment_outcomes.mean()
obs_lift      = obs_treatment - obs_control

print("=" * 55)
print("  SIMULATED A/B TEST — 7-DAY DEFAULT BROWSER SET")
print("=" * 55)
print(f"\n  Devices per variant      : {n_per_variant:,}")
print(f"\n  {'Group':<12}  {'Set default':>12}  {'Rate':>8}")
print(f"  {'-'*36}")
print(f"  {'Control':<12}  {control_outcomes.sum():>12,}  {obs_control:>8.2%}")
print(f"  {'Treatment':<12}  {treatment_outcomes.sum():>12,}  {obs_treatment:>8.2%}")
print(f"\n  Observed lift            : {obs_lift:+.2%}")
print(f"  True lift                : {true_lift:+.2%}")
print(f"  Sampling noise           : {obs_lift - true_lift:+.2%}")
print("=" * 55)


  SIMULATED A/B TEST — 14-DAY PB RETENTION

  Devices per variant      : 17,210

  Group           Retained      Rate
  ----------------------------------
  Control            9,439    54.85%
  Treatment          9,703    56.38%

  Observed lift            : +1.53%
  True lift                : +1.50%
  Sampling noise           : +0.03%


### Statistical Test & Confidence Interval


In [28]:
from statsmodels.stats.proportion import proportions_ztest
from scipy import stats

mde = 0.015
alpha = 0.05

counts = np.array([treatment_outcomes.sum(), control_outcomes.sum()])
nobs   = np.array([len(treatment_outcomes), len(control_outcomes)])

z_stat, p_value = proportions_ztest(counts, nobs, alternative='two-sided')

# 95% CI for the difference in proportions (treatment - control)
p_c  = control_outcomes.mean()
p_t  = treatment_outcomes.mean()
diff = p_t - p_c
se   = np.sqrt(p_c * (1 - p_c) / len(control_outcomes) + p_t * (1 - p_t) / len(treatment_outcomes))
z_crit   = stats.norm.ppf(1 - alpha / 2)
ci_lower = diff - z_crit * se
ci_upper = diff + z_crit * se

print("=" * 55)
print("  TWO-PROPORTION Z-TEST — 7-DAY DEFAULT BROWSER SET")
print("=" * 55)
print(f"\n  Control default-set rate : {p_c:.2%}  (n={len(control_outcomes):,})")
print(f"  Treatment default-set rate: {p_t:.2%}  (n={len(treatment_outcomes):,})")
print(f"\n  Z-statistic              : {z_stat:.4f}")
print(f"  P-value (two-tailed)     : {p_value:.4f}")
print(f"\n  95% CI for lift          : ({ci_lower:+.4f}, {ci_upper:+.4f})")
print(f"                             ({ci_lower*100:+.2f}pp, {ci_upper*100:+.2f}pp)")
print(f"\n  {'─'*45}")

sig          = p_value < alpha
mde_exceeded = ci_lower >= mde

print(f"\n  Significant (p < {alpha})?       {'YES' if sig else 'NO'}")
print(f"  CI lower bound ≥ MDE (1.5pp)?  {'YES' if mde_exceeded else 'NO'}")
print(f"\n  Interpretation:")
if sig and mde_exceeded:
    print("  Reject H₀. Default-set rate increased significantly")
    print("  and CI clears the MDE — strong case to ship.")
    print("  Monitor share of browsing time at day 30 to confirm durability.")
elif sig and not mde_exceeded:
    print("  Reject H₀. Lift is significant but CI lower bound is below")
    print("  the 1.5pp MDE — statistically present but practically uncertain.")
    print("  Hold for a larger sample or discuss whether sub-MDE lift")
    print("  justifies the UX change.")
else:
    print("  Fail to reject H₀. No significant lift in default-set rate.")
    print("  Do not ship on this read.")
print("=" * 55)


  TWO-PROPORTION Z-TEST RESULTS

  Control retention        : 54.85%  (n=17,210)
  Treatment retention      : 56.38%  (n=17,210)

  Z-statistic              : 2.8641
  P-value (two-tailed)     : 0.0042

  95% CI for lift          : (+0.0048, +0.0258)
                             (+0.48pp, +2.58pp)

  ─────────────────────────────────────────────

  Significant (p < 0.05)?       YES
  CI lower bound ≥ MDE (1.5pp)?  NO

  Interpretation:
  Reject H₀. Lift is significant.
CI lower bound is below the 1.5pp MDE — statistically present but practically uncertain. Hold for a larger sample or discuss whether sub-MDE lift justifies the UX change.


#### Guardrail Metrics

In [ ]:
from scipy import stats
import numpy as np

rng_g = np.random.default_rng(seed=99)

# Search queries per device — hard guardrail
# If Copilot answers questions users would have searched for, query volume drops
# Direct revenue risk — search query drop = ad revenue loss

# Baseline: ~12 search queries per device per day
# Treatment: slight drop simulated — Copilot may substitute for some searches
search_control   = rng_g.normal(loc=12.0, scale=4.0, size=n_per_variant)
search_treatment = rng_g.normal(loc=11.6, scale=4.0, size=n_per_variant)

t_stat, p_val = stats.ttest_ind(search_control, search_treatment)
diff = search_treatment.mean() - search_control.mean()
flag = "FLAGGED" if p_val < 0.05 else "ok"

print("Search Guardrail Check")
print("-" * 45)
print(f"Control   mean queries/device : {search_control.mean():.2f}")
print(f"Treatment mean queries/device : {search_treatment.mean():.2f}")
print(f"Difference                    : {diff:+.2f}")
print(f"T-statistic                   : {t_stat:.4f}")
print(f"P-value                       : {p_val:.4f}")
print(f"Result                        : [{flag}]")
print()
if p_val < 0.05:
    print("Search queries dropped significantly in treatment.")
    print("Even if default-set rate improves, this is a revenue tradeoff.")
    print("Requires business decision — not an automatic ship.")
else:
    print("No significant drop in search queries. Guardrail holds.")


Search Guardrail Check
---------------------------------------------
Control   mean queries/device : 12.02
Treatment mean queries/device : 11.58
Difference                    : -0.44
T-statistic                   : 10.1579
P-value                       : 0.0000
Result                        : [FLAGGED]

Search queries dropped significantly in treatment.
Even if PB retention improves, this is a revenue tradeoff.
Requires business decision — not an automatic ship.


#### SRM Check

In simulation, assignment is exact (17,210 per variant by construction) so assignment imbalance cannot occur here. In a real experiment, checking assignment balance is the first step before reading any results — chi-square goodness-of-fit on observed vs expected assignment counts. Common causes of imbalance: logging failures, bot filtering on one arm only, bucketing bugs in the assignment pipeline. If detected, experiment results are invalid regardless of what the primary metric shows.


#### Novelty Effect

A contextual prompt may produce inflated engagement in the first few days as users explore the new surface. Reading default browser set rate at day 7 — rather than day 3 — allows the initial excitement to settle and captures conversions that happen after session 2 or 3. If the treatment effect is strong at day 7 but default retention drops sharply by day 30, that is a signal the default-set was novelty-driven rather than habit-driven.


#### Next Steps

- ITT vs CACE: current analysis is intent-to-treat. Next step would be estimating the complier average causal effect — among devices the nudge actually moved to use Copilot, what was the effect on default-set rate?
- Segment analysis: does the effect differ by device type, market, or session frequency?
- Day 30 durability read: share of browsing time and default retention rate once post-ship monitoring window closes
- Session of first exposure as covariate — controls for timing confound
